<a href="https://colab.research.google.com/github/Cheikhcisse1/projet_FN/blob/main/SerieTemporelle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Les bibiothéques Besion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import boxcox

In [ ]:
class SerieTemporelle:

    def __init__(self, donnee, colonne, date='date'):
        """
        Initialise la classe avec un DataFrame.

        donnee : DataFrame contenant les données
        colonne : nom de la variable à analyser
        date : nom de la colonne contenant les dates
        """

        self.donnee = donnee.copy()
        self.colonne = colonne
        self.date = date

        if self.colonne not in self.donnee.columns:
            raise ValueError(f"La colonne '{self.colonne}' n'existe pas.")

        if self.date not in self.donnee.columns:
            raise ValueError(f"La colonne '{self.date}' n'existe pas.")

        self.donnee[self.date] = pd.to_datetime(
            self.donnee[self.date], errors='coerce'
        )

        self.donnee = self.donnee.dropna(subset=[self.date])
        self.donnee = self.donnee.sort_values(self.date)
        self.donnee = self.donnee.set_index(self.date)

        self.donnee[self.colonne] = pd.to_numeric(
            self.donnee[self.colonne], errors='coerce'
        )

    # --------------------------------------------------
    # 1. AFFICHAGE ET DESCRIPTION DES DONNEES
    # --------------------------------------------------

    def traitementDes_donne(self):
        print("Dimensions des données :", self.donnee.shape)
        print("\nInformations générales :")
        print(self.donnee.info())

        print("\nStatistiques descriptives :")
        print(self.donnee.describe())

        print("\nValeurs manquantes :")
        print(self.donnee.isnull().sum())

        print("\nDoublons dans l'index :", self.donnee.index.duplicated().sum())

    # --------------------------------------------------
    # 2. VISUALISATION DE LA SERIE
    # --------------------------------------------------

    def visualisation(self):
        plt.figure(figsize=(12, 5))

        plt.plot(
            self.donnee.index,
            self.donnee[self.colonne],
            label=self.colonne
        )

        plt.title(f"Évolution de {self.colonne}")
        plt.xlabel("Date")
        plt.ylabel(self.colonne)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    # --------------------------------------------------
    # 3. TRAITEMENT DES VALEURS MANQUANTES
    # --------------------------------------------------

    def traitementValeurmanquantes(self, methode='interpolate'):
        """
        Méthodes disponibles :
        - interpolate
        - mean
        - median
        - ffill
        - bfill
        - drop
        """

        nombre = self.donnee[self.colonne].isnull().sum()

        print("Nombre de valeurs manquantes :", nombre)

        if methode == 'interpolate':
            self.donnee[self.colonne] = (
                self.donnee[self.colonne].interpolate(method='time')
            )

        elif methode == 'mean':
            self.donnee[self.colonne] = (
                self.donnee[self.colonne].fillna(
                    self.donnee[self.colonne].mean()
                )
            )

        elif methode == 'median':
            self.donnee[self.colonne] = (
                self.donnee[self.colonne].fillna(
                    self.donnee[self.colonne].median()
                )
            )

        elif methode == 'ffill':
            self.donnee[self.colonne] = (
                self.donnee[self.colonne].ffill()
            )

        elif methode == 'bfill':
            self.donnee[self.colonne] = (
                self.donnee[self.colonne].bfill()
            )

        elif methode == 'drop':
            self.donnee = self.donnee.dropna(
                subset=[self.colonne]
            )

        else:
            raise ValueError("Méthode de traitement inconnue.")

        print(
            "Valeurs manquantes restantes :",
            self.donnee[self.colonne].isnull().sum()
        )

        return self.donnee

    # --------------------------------------------------
    # 4. PREPARATION DE L'INDEX TEMPOREL
    # --------------------------------------------------

    def preparer_index(self, frequence=None):
        """
        Définit une fréquence temporelle si elle est connue.
        Exemples : 'D' (jour), 'W' (semaine), 'MS' (mois).
        """

        self.donnee = self.donnee.sort_index()

        if self.donnee.index.has_duplicates:
            raise ValueError(
                "L'index contient des dates en double. "
                "Il faut les agréger avant de continuer."
            )

        if frequence is not None:
            self.donnee = self.donnee.asfreq(frequence)

        else:
            frequence_detectee = pd.infer_freq(self.donnee.index)

            if frequence_detectee is not None:
                self.donnee = self.donnee.asfreq(frequence_detectee)
                print("Fréquence détectée :", frequence_detectee)
            else:
                print(
                    "Fréquence non détectée. "
                    "Précisez-la si nécessaire."
                )

        return self.donnee

    # --------------------------------------------------
    # 5. DECOMPOSITION DE LA SERIE TEMPORELLE
    # --------------------------------------------------

    def decomposition(self, periode, modele='additive'):
        serie = self.donnee[self.colonne].dropna()

        resultat = seasonal_decompose(
            serie,
            model=modele,
            period=periode
        )

        resultat.plot()
        plt.suptitle(
            f"Décomposition de {self.colonne}",
            fontsize=14
        )
        plt.tight_layout()
        plt.show()

        return resultat

    # --------------------------------------------------
    # 6. TEST DE STATIONNARITE ADF
    # --------------------------------------------------

    def ADF_test(self, colonne=None):
        if colonne is None:
            colonne = self.colonne

        serie = self.donnee[colonne].dropna()

        result = adfuller(serie)

        print("\n===== TEST ADF =====")
        print("ADF Statistic :", result[0])
        print("p-value :", result[1])
        print("Nombre de retards :", result[2])
        print("Nombre d'observations :", result[3])

        if result[1] > 0.05:
            print("Conclusion : la série n'est pas stationnaire.")
        else:
            print("Conclusion : la série est stationnaire.")

        print("\nValeurs critiques :")
        for key, value in result[4].items():
            print(f"{key} : {value:.3f}")

        return result

    # --------------------------------------------------
    # 7. TEST DE STATIONNARITE KPSS
    # --------------------------------------------------

    def KPSS_test(self, colonne=None):
        if colonne is None:
            colonne = self.colonne

        serie = self.donnee[colonne].dropna()

        result = kpss(
            serie,
            regression='c',
            nlags='auto'
        )

        print("\n===== TEST KPSS =====")
        print("KPSS Statistic :", result[0])
        print("p-value :", result[1])
        print("Nombre de retards :", result[2])

        if result[1] < 0.05:
            print("Conclusion : la série n'est pas stationnaire.")
        else:
            print(
                "Conclusion : on ne rejette pas "
                "l'hypothèse de stationnarité."
            )

        print("\nValeurs critiques :")
        for key, value in result[3].items():
            print(f"{key} : {value:.3f}")

        return result

    # --------------------------------------------------
    # 8. COURBES ACF ET PACF
    # --------------------------------------------------

    def courbe_ACF_PACF(self, lags=40, colonne=None):
        if colonne is None:
            colonne = self.colonne

        serie = self.donnee[colonne].dropna()

        if len(serie) <= lags:
            lags = max(1, len(serie) // 2 - 1)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        plot_acf(serie, lags=lags, ax=axes[0])
        axes[0].set_title("Autocorrelation (ACF)")

        plot_pacf(
            serie,
            lags=lags,
            ax=axes[1],
            method='ywm'
        )
        axes[1].set_title("Autocorrelation partielle (PACF)")

        plt.tight_layout()
        plt.show()

    # --------------------------------------------------
    # 9. DIFFERENCIATION
    # --------------------------------------------------

    def differenciation(self, ordre=1, saisonniere=False,
                        periode=12):
        """
        ordre : ordre de différenciation
        saisonniere : True pour une différenciation saisonnière
        periode : période saisonnière (12 pour données mensuelles)
        """

        serie = self.donnee[self.colonne]

        if saisonniere:
            self.donnee[self.colonne + '_diff'] = (
                serie.diff(periode)
            )
        else:
            self.donnee[self.colonne + '_diff'] = (
                serie.diff(ordre)
            )

        nom_colonne = self.colonne + '_diff'

        plt.figure(figsize=(12, 5))
        plt.plot(
            self.donnee.index,
            self.donnee[nom_colonne],
            label='Série différenciée'
        )

        plt.title(f"Différenciation de {self.colonne}")
        plt.xlabel("Date")
        plt.ylabel("Valeur différenciée")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        return self.donnee[nom_colonne].dropna()

    # --------------------------------------------------
    # 10. TRANSFORMATION LOGARITHMIQUE
    # --------------------------------------------------

    def logarithme(self):
        serie = self.donnee[self.colonne]

        if (serie.dropna() <= 0).any():
            raise ValueError(
                "La transformation logarithmique nécessite "
                "des valeurs strictement positives."
            )

        self.donnee[self.colonne + '_log'] = np.log(serie)

        plt.figure(figsize=(12, 5))
        plt.plot(
            self.donnee.index,
            self.donnee[self.colonne + '_log']
        )

        plt.title(f"Transformation logarithmique de {self.colonne}")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        return self.donnee[self.colonne + '_log']

    # --------------------------------------------------
    # 11. TRANSFORMATION BOX-COX
    # --------------------------------------------------

    def boxscox(self):
        serie = self.donnee[self.colonne]

        valeurs = serie.dropna()

        if (valeurs <= 0).any():
            raise ValueError(
                "La transformation Box-Cox nécessite "
                "des valeurs strictement positives."
            )

        valeurs_transformees, lambda_boxcox = boxcox(valeurs)

        nom_colonne = self.colonne + '_boxcox'

        self.donnee[nom_colonne] = np.nan
        self.donnee.loc[valeurs.index, nom_colonne] = valeurs_transformees

        print("Lambda optimal :", lambda_boxcox)

        plt.figure(figsize=(12, 5))
        plt.plot(
            self.donnee.index,
            self.donnee[nom_colonne]
        )

        plt.title(f"Transformation Box-Cox de {self.colonne}")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        return self.donnee[nom_colonne], lambda_boxcox


    def rendre_stationnaire(self, periode=12):
        # 1. Transformation logarithmique
        serie_log = np.log(self.donnee[self.colonne])

        # 2. Différenciation simple
        serie_diff = serie_log.diff()

        # 3. Différenciation saisonnière
        serie_stationnaire = serie_diff.diff(periode)

        # Enregistrement dans le DataFrame
        nom_colonne = self.colonne + '_stationnaire'
        self.donnee[nom_colonne] = serie_stationnaire

        # Visualisation
        plt.figure(figsize=(12, 5))
        plt.plot(
            self.donnee.index,
            self.donnee[nom_colonne],
            label='Série stationnaire'
        )
        plt.title('Série après différenciation simple et saisonnière')
        plt.xlabel('Date')
        plt.ylabel('Valeur transformée')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        # Tests de stationnarité
        print("\n===== TEST ADF =====")
        self.ADF_test(nom_colonne)

        print("\n===== TEST KPSS =====")
        self.KPSS_test(nom_colonne)

        return self.donnee[nom_colonne].dropna()



In [ ]:

if __name__ == "__main__":

    chemin = '/content/aqi_mensuel.csv'

    # Étape 1 : Analyse exploratoire et visualisation initiale
    donnee_base = pd.read_csv(chemin)

    donnee = SerieTemporelle(
        donnee_base.copy(),
        colonne='aqi',
        date='date'
    )

    donnee.traitementDes_donne()
    donnee.visualisation()

    # Étape 2 : Traitement des valeurs manquantes
    donnee.traitementValeurmanquantes(methode='interpolate')

    # Étape 3 : Préparation de l'index temporel
    donnee.preparer_index(frequence='MS')

    # Étape 4 : Décomposition de la série temporelle
    print("\n===== DÉCOMPOSITION =====")

    df_decomp = donnee.decomposition(
        periode=12,
        modele='additive'
    )

    print(df_decomp.trend.head(12))
    print(df_decomp.seasonal.head(12))
    print(df_decomp.resid.head(12))

    print('********************************' * 5)

    # Étape 5 : Test de stationnarité ADF
    print("\n===== TEST ADF =====")
    donnee.ADF_test()

    print("---------------------------------------------------")

    # Étape 6 : Test de stationnarité KPSS
    print("\n===== TEST KPSS =====")
    donnee.KPSS_test()

    print("---------------------------------------------------")

    # Étape 7 : Graphiques ACF et PACF
    print("\n===== ACF ET PACF =====")
    donnee.courbe_ACF_PACF(lags=24)


In [ ]:
donnee.differenciation()

In [ ]:
donnee.logarithme()

In [ ]:
import scipy.stats as stats
donnee.boxscox()

In [ ]:
#!pip install skforecast
!pip install pmdarima

In [ ]:
from skforecast.stats import Arima, Sarimax
from pmdarima import auto_arima
from skforecast.model_selection import backtesting_forecaster
from skforecast.model_selection import grid_search_forecaster

class Sarimax_model(SerieTemporelle):
  def __init__(self, donnee, p, d, q, P, D, Q, s):
    super().__init__(donnee)
    self.p = p
    self.d = d
    self.q = q
    self.P = P
    self.D = D
    self.Q = Q
    self.s = s

  def entrainement_sarimax(self):
    self.preparer_index()
    self.model = Sarimax(order=(self.p, self.d, self.q), seasonal_order=(self.P, self.D, self.Q, self.s))
    self.model

  def entrainement_arima(self):
    self.preparer_index()
    self.model = Arima(order=(self.p, self.d, self.q))
    self.model

In [ ]:
import statsmodels.api as sm
import pandas as pd

if __name__ == "__main__":

    # Chargement des données AirPassengers
    data = sm.datasets.get_rdataset("AirPassengers", "datasets")

    dates = pd.date_range(
        start='1949-01',
        periods=144,
        freq='MS'
    )

    serie = pd.Series(
        data.data['value'].values,
        index=dates,
        name='passagers'
    )

    # Conversion en DataFrame
    data_df = serie.reset_index()
    data_df.columns = ['date', 'aqi']

    # Initialisation de la classe
    donneee = SerieTemporelle(
        data_df,
        colonne='aqi',
        date='date'
    )

    # Étape 1 : Analyse exploratoire
    donneee.traitementDes_donne()
    donneee.visualisation()

    # Étape 2 : Traitement des valeurs manquantes
    donneee.traitementValeurmanquantes(
        methode='interpolate'
    )

    # Étape 3 : Décomposition de la série
    print("\n===== DÉCOMPOSITION =====")

    df_decomp = donneee.decomposition(
        periode=12,
        modele='multiplicative'
    )

    # Étape 4 : Test ADF
    print("\n===== TEST ADF =====")
    donneee.ADF_test()

    # Étape 5 : Test KPSS
    print("\n===== TEST KPSS =====")
    donneee.KPSS_test()

    # Étape 6 : ACF et PACF
    print("\n===== ACF ET PACF =====")
    donneee.courbe_ACF_PACF(lags=24)


In [ ]:

# Rendre la série stationnaire
print("\n===== STATIONNARISATION =====")

serie_stationnaire = donneee.rendre_stationnaire(periode=12)

# Visualisation ACF et PACF de la série stationnaire
donneee.courbe_ACF_PACF(
    lags=24,
    colonne='aqi_stationnaire'
)
